# Revised ADNI age / ROC / threshold / deployment-audit analysis

This notebook replaces the older `delong_permutation.ipynb` workflow for the TMLR revision.

It uses the revised subject-level OOF prediction table with scan-level age matching:

```text
results/autorater/autorater_predictions_all_subject_oof_with_scan_age.csv
```

Main outputs:

- overall and age-stratified ROC/AUC
- pairwise age-bin AUC permutation tests with Holm correction
- fixed-threshold and Youden-threshold operating points
- calibration/Brier diagnostics
- real-data prevalence estimates using PPI, PPI++, CRE Uniform, and CRE Jeffreys
- subject-grouped OOF threshold-selection audit vs leaky full-data threshold
- labeled/unlabeled exchangeability diagnostic
- bootstrap dispersion of Youden threshold

In [ ]:
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


In [ ]:
%pip install -q pandas numpy scipy scikit-learn matplotlib tqdm

In [ ]:
from pathlib import Path
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import t as student_t
from tqdm import tqdm

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    auc,
    accuracy_score,
    confusion_matrix,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
# ============================================================
# 0) Paths and settings
# ============================================================

CSV_PRED = REPO_ROOT / "results/autorater/autorater_predictions_all_subject_oof_with_scan_age.csv"

OUT_DIR = REPO_ROOT / "results/case_study"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CAL_DIR = OUT_DIR / "calibration_curves"
CAL_DIR.mkdir(parents=True, exist_ok=True)

AGE_BINS = [50, 74, 80, 101]
AGE_LABELS = ["50-73", "74-79", "80-100"]

# Exact scan-level subset requested by reviewer discussion
EXACT_6570_LOW = 65
EXACT_6570_HIGH = 70

THRESHOLD = 0.5
SEED = 2025

BOOTSTRAP_B = 2000
PERM_B = 5000
THRESH_BOOT_B = 2000

Z975 = 1.959963984540054

rng = np.random.default_rng(SEED)

print("Input:", CSV_PRED)
print("Output:", OUT_DIR)

In [ ]:
# ============================================================
# 1) Load and QC revised prediction table
# ============================================================

def load_revised_prediction_table(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"Could not find input CSV: {csv_path}")

    df = pd.read_csv(csv_path).copy()
    df.columns = [c.strip() for c in df.columns]

    required = {"subject_id", "H", "Age", "autorater_prediction", "oof_fold"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df["subject_id"] = df["subject_id"].astype(str).str.strip()
    df["H"] = pd.to_numeric(df["H"], errors="coerce").astype(int)
    df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
    df["A_prob"] = pd.to_numeric(df["autorater_prediction"], errors="coerce")
    df["A_class"] = (df["A_prob"] >= THRESHOLD).astype(int)

    df = df.dropna(subset=["subject_id", "H", "Age", "A_prob", "oof_fold"]).copy()
    df = df[df["A_prob"].between(0, 1, inclusive="both")].copy()
    df["oof_fold"] = pd.to_numeric(df["oof_fold"], errors="coerce").astype(int)

    # Age bins: [50, 74), [74, 80), [80, 101)
    df["age_bin"] = pd.cut(
        df["Age"],
        bins=AGE_BINS,
        labels=AGE_LABELS,
        right=False,
        include_lowest=True,
    )

    df["is_age_65_70"] = (df["Age"] >= EXACT_6570_LOW) & (df["Age"] <= EXACT_6570_HIGH)

    return df.reset_index(drop=True)


use = load_revised_prediction_table(CSV_PRED)

print("=" * 90)
print("QC: revised prediction table")
print("=" * 90)
print("Rows:", len(use))
print("Unique subjects:", use["subject_id"].nunique())
print("True prevalence:", float(use["H"].mean()))
print("P(A=1) @ 0.5:", float(use["A_class"].mean()))
print("Age range:", float(use["Age"].min()), "to", float(use["Age"].max()))
print("Age missing:", int(use["Age"].isna().sum()))
print("Age bins:")
print(use["age_bin"].value_counts(dropna=False).sort_index())

per_subject_fold_n = use.groupby("subject_id")["oof_fold"].nunique()
print("Subjects spanning multiple OOF folds:", int((per_subject_fold_n > 1).sum()))
print("Max folds per subject:", int(per_subject_fold_n.max()))
print("OOF fold row counts:")
print(use["oof_fold"].value_counts().sort_index())

y = use["H"].to_numpy(dtype=int)
p = use["A_prob"].to_numpy(dtype=float)
a = use["A_class"].to_numpy(dtype=int)

tn, fp, fn, tp = confusion_matrix(y, a, labels=[0, 1]).ravel()

print("\nAutorater metrics")
print("AUC:", float(roc_auc_score(y, p)))
print("ACC @ 0.5:", float(accuracy_score(y, a)))
print("TN, FP, FN, TP:", int(tn), int(fp), int(fn), int(tp))
print("TPR:", float(tp / (tp + fn)) if (tp + fn) else np.nan)
print("TNR:", float(tn / (tn + fp)) if (tn + fp) else np.nan)

use.to_csv(OUT_DIR / "revised_prediction_table_qc_copy.csv", index=False)
display(use.head())

In [ ]:
# ============================================================
# 2) Utility functions
# ============================================================

def wilson_ci(k: int, n: int, z: float = 1.959963984540054):
    if n <= 0:
        return (np.nan, np.nan)
    phat = k / n
    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    half = z * np.sqrt((phat * (1 - phat) / n) + z**2 / (4 * n**2)) / denom
    return float(center - half), float(center + half)


def binary_metrics(y_true, score, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    yhat = (score >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, yhat, labels=[0, 1]).ravel()

    acc = accuracy_score(y_true, yhat)
    tpr = tp / (tp + fn) if (tp + fn) else np.nan
    tnr = tn / (tn + fp) if (tn + fp) else np.nan

    acc_lo, acc_hi = wilson_ci(int((yhat == y_true).sum()), len(y_true))
    tpr_lo, tpr_hi = wilson_ci(int(tp), int(tp + fn))
    tnr_lo, tnr_hi = wilson_ci(int(tn), int(tn + fp))

    return {
        "ACC": float(acc),
        "ACC_lo": acc_lo,
        "ACC_hi": acc_hi,
        "TPR": float(tpr),
        "TPR_lo": tpr_lo,
        "TPR_hi": tpr_hi,
        "TNR": float(tnr),
        "TNR_lo": tnr_lo,
        "TNR_hi": tnr_hi,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }


def bootstrap_auc_ci(y_true, score, B=2000, seed=2025):
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)

    auc0 = roc_auc_score(y_true, score)
    local_rng = np.random.default_rng(seed)

    vals = []
    n = len(y_true)

    for _ in range(B):
        idx = local_rng.integers(0, n, size=n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        vals.append(roc_auc_score(y_true[idx], score[idx]))

    if len(vals) == 0:
        return float(auc0), np.nan, np.nan

    lo, hi = np.quantile(vals, [0.025, 0.975])
    return float(auc0), float(lo), float(hi)


def plot_roc_curve(y_true, score, title, save_path):
    fpr, tpr, _ = roc_curve(y_true, score)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(5.5, 5.0))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    return float(roc_auc)


def youden_threshold(y_true, score):
    fpr, tpr, thresholds = roc_curve(y_true, score)
    j = tpr - fpr
    k = int(np.argmax(j))
    threshold = thresholds[k]
    return float(threshold), float(tpr[k]), float(1.0 - fpr[k])

In [ ]:
# ============================================================
# 3) Overall ROC and age-stratified metrics
# ============================================================

# Overall ROC
overall_auc = plot_roc_curve(
    use["H"].values,
    use["A_prob"].values,
    "Overall ROC: subject-level OOF",
    FIG_DIR / "fig_roc_overall_subject_oof.png",
)

rows = []

# Overall row
auc0, auc_lo, auc_hi = bootstrap_auc_ci(use["H"].values, use["A_prob"].values, B=BOOTSTRAP_B, seed=SEED)
m = binary_metrics(use["H"].values, use["A_prob"].values, threshold=THRESHOLD)
rows.append({
    "scope": "Overall",
    "N": len(use),
    "subjects": use["subject_id"].nunique(),
    "prevalence": float(use["H"].mean()),
    "AUC": auc0,
    "AUC_lo": auc_lo,
    "AUC_hi": auc_hi,
    **m,
})

# Age-bin rows
for lab in AGE_LABELS:
    sub = use[use["age_bin"] == lab].copy()
    if len(sub) == 0 or sub["H"].nunique() < 2:
        continue

    auc0, auc_lo, auc_hi = bootstrap_auc_ci(sub["H"].values, sub["A_prob"].values, B=BOOTSTRAP_B, seed=SEED + len(rows))
    m = binary_metrics(sub["H"].values, sub["A_prob"].values, threshold=THRESHOLD)

    rows.append({
        "scope": lab,
        "N": len(sub),
        "subjects": sub["subject_id"].nunique(),
        "prevalence": float(sub["H"].mean()),
        "AUC": auc0,
        "AUC_lo": auc_lo,
        "AUC_hi": auc_hi,
        **m,
    })

    plot_roc_curve(
        sub["H"].values,
        sub["A_prob"].values,
        f"ROC: Age {lab}",
        FIG_DIR / f"fig_roc_age_{lab.replace('-', '_')}.png",
    )

# Exact 65--70 row
sub6570 = use[use["is_age_65_70"]].copy()
auc0, auc_lo, auc_hi = bootstrap_auc_ci(sub6570["H"].values, sub6570["A_prob"].values, B=BOOTSTRAP_B, seed=SEED + 999)
m = binary_metrics(sub6570["H"].values, sub6570["A_prob"].values, threshold=THRESHOLD)
rows.append({
    "scope": "Age 65-70",
    "N": len(sub6570),
    "subjects": sub6570["subject_id"].nunique(),
    "prevalence": float(sub6570["H"].mean()),
    "AUC": auc0,
    "AUC_lo": auc_lo,
    "AUC_hi": auc_hi,
    **m,
})

plot_roc_curve(
    sub6570["H"].values,
    sub6570["A_prob"].values,
    "ROC: Age 65--70 subset",
    FIG_DIR / "fig_roc_age_65_70.png",
)

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(OUT_DIR / "metrics_overall_age_auc_ci.csv", index=False)

display(metrics_df.round(4))

In [ ]:
# ============================================================
# 4) Overlay ROC curves by age bins
# ============================================================

plt.figure(figsize=(6.2, 5.2))

for lab in AGE_LABELS:
    sub = use[use["age_bin"] == lab].copy()
    if len(sub) == 0 or sub["H"].nunique() < 2:
        continue

    fpr, tpr, _ = roc_curve(sub["H"].values, sub["A_prob"].values)
    auc_val = roc_auc_score(sub["H"].values, sub["A_prob"].values)
    plt.plot(fpr, tpr, label=f"{lab}, AUC={auc_val:.3f}, n={len(sub)}")

plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("Age-stratified ROC curves")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_roc_by_age_bins_subject_oof.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 5) Pairwise age-bin AUC permutation tests with Holm correction
# ============================================================

def _auc_np(y, p):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    if len(np.unique(y)) < 2:
        return np.nan
    return roc_auc_score(y, p)


def perm_test_auc_diff(df, bin1, bin2, B=5000, seed=2025):
    """
    Two-sided permutation test for difference in AUC between two independent age bins.
    Null: age-bin group labels are exchangeable.
    """
    d1 = df[df["age_bin"] == bin1].copy()
    d2 = df[df["age_bin"] == bin2].copy()

    y1 = d1["H"].to_numpy(dtype=int)
    p1 = d1["A_prob"].to_numpy(dtype=float)
    y2 = d2["H"].to_numpy(dtype=int)
    p2 = d2["A_prob"].to_numpy(dtype=float)

    auc1 = _auc_np(y1, p1)
    auc2 = _auc_np(y2, p2)
    obs = auc1 - auc2

    y_all = np.concatenate([y1, y2])
    p_all = np.concatenate([p1, p2])
    n1 = len(y1)

    local_rng = np.random.default_rng(seed)
    vals = []

    for _ in range(B):
        idx = local_rng.permutation(len(y_all))
        idx1 = idx[:n1]
        idx2 = idx[n1:]

        auc1_b = _auc_np(y_all[idx1], p_all[idx1])
        auc2_b = _auc_np(y_all[idx2], p_all[idx2])

        if np.isfinite(auc1_b) and np.isfinite(auc2_b):
            vals.append(auc1_b - auc2_b)

    vals = np.asarray(vals)
    pval = float(np.mean(np.abs(vals) >= abs(obs))) if len(vals) else np.nan

    return {
        "bin1": bin1,
        "bin2": bin2,
        "n1": len(d1),
        "n2": len(d2),
        "auc1": float(auc1),
        "auc2": float(auc2),
        "auc_diff": float(obs),
        "p_raw": pval,
        "B_used": int(len(vals)),
    }


def holm_adjust(pvals):
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)

    adj = np.empty(m, dtype=float)
    running_max = 0.0

    for rank, idx in enumerate(order):
        adjusted = (m - rank) * pvals[idx]
        running_max = max(running_max, adjusted)
        adj[idx] = min(running_max, 1.0)

    return adj


pair_rows = []
pairs = list(itertools.combinations(AGE_LABELS, 2))

for j, (b1, b2) in enumerate(pairs):
    pair_rows.append(perm_test_auc_diff(use, b1, b2, B=PERM_B, seed=SEED + 100 + j))

pairwise_df = pd.DataFrame(pair_rows)
if len(pairwise_df):
    pairwise_df["p_holm"] = holm_adjust(pairwise_df["p_raw"].values)
    pairwise_df["method"] = "Two-sided permutation test with Holm correction"

pairwise_df.to_csv(OUT_DIR / "pairwise_auc_perm_test_subject_oof.csv", index=False)
display(pairwise_df.round(4))

In [ ]:
# ============================================================
# 6) Fixed threshold and Youden threshold operating points
# ============================================================

rows = []

scopes = [("Overall", use.copy())]
for lab in AGE_LABELS:
    scopes.append((lab, use[use["age_bin"] == lab].copy()))
scopes.append(("Age 65-70", use[use["is_age_65_70"]].copy()))

for scope, df_scope in scopes:
    if len(df_scope) == 0 or df_scope["H"].nunique() < 2:
        continue

    y = df_scope["H"].to_numpy(dtype=int)
    p = df_scope["A_prob"].to_numpy(dtype=float)

    # Fixed 0.5
    fixed_metrics = binary_metrics(y, p, threshold=0.5)
    rows.append({
        "scope": scope,
        "threshold_policy": "fixed_0.5",
        "threshold": 0.5,
        "N": len(df_scope),
        "subjects": df_scope["subject_id"].nunique(),
        "prevalence": float(df_scope["H"].mean()),
        "AUC": float(roc_auc_score(y, p)),
        **fixed_metrics,
    })

    # Youden threshold on same scope, leaky descriptive operating point
    thr, youden_tpr, youden_tnr = youden_threshold(y, p)
    y_metrics = binary_metrics(y, p, threshold=thr)
    rows.append({
        "scope": scope,
        "threshold_policy": "youden_same_scope_descriptive",
        "threshold": thr,
        "N": len(df_scope),
        "subjects": df_scope["subject_id"].nunique(),
        "prevalence": float(df_scope["H"].mean()),
        "AUC": float(roc_auc_score(y, p)),
        **y_metrics,
    })

threshold_metrics_df = pd.DataFrame(rows)
threshold_metrics_df.to_csv(OUT_DIR / "threshold_metrics_fixed_and_youden.csv", index=False)

display(threshold_metrics_df.round(4))

In [ ]:
# ============================================================
# 7) Calibration / Brier diagnostics
# ============================================================

cal_rows = []

for scope, df_scope in scopes:
    if len(df_scope) < 50 or df_scope["H"].nunique() < 2:
        continue

    y = df_scope["H"].to_numpy(dtype=int)
    p = df_scope["A_prob"].to_numpy(dtype=float)

    frac_pos, mean_pred = calibration_curve(y, p, n_bins=10, strategy="uniform")
    bs = brier_score_loss(y, p)

    plt.figure(figsize=(5.2, 5.0))
    plt.plot(mean_pred, frac_pos, "o-", label="Empirical")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Fraction positive")
    plt.title(f"Calibration: {scope}, Brier={bs:.3f}")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    safe_scope = scope.replace(" ", "_").replace("-", "_")
    plt.savefig(CAL_DIR / f"calibration_{safe_scope}.png", dpi=300, bbox_inches="tight")
    plt.show()

    cal_rows.append({
        "scope": scope,
        "N": len(df_scope),
        "subjects": df_scope["subject_id"].nunique(),
        "Brier": float(bs),
    })

calibration_df = pd.DataFrame(cal_rows)
calibration_df.to_csv(OUT_DIR / "calibration_brier_by_scope.csv", index=False)

display(calibration_df.round(4))

In [ ]:
# ============================================================
# 8) Real-data prevalence estimates: PPI, PPI++, CRE
# ============================================================

def ppi_analytic_estimator(df_all):
    """
    Prior-free continuous-score PPI using all rows as labeled for descriptive real-data estimate.
    g_hat = mean(p_all) + mean(H - p)
    """
    p_all = df_all["A_prob"].to_numpy(dtype=float)
    H = df_all["H"].to_numpy(dtype=float)

    N = len(df_all)
    resid = H - p_all

    g_hat = float(p_all.mean() + resid.mean())

    var_p = float(np.var(p_all, ddof=1)) if N > 1 else 0.0
    var_r = float(np.var(resid, ddof=1)) if N > 1 else 0.0

    se = np.sqrt(var_p / N + var_r / N)
    crit = Z975 if N >= 30 else float(student_t.ppf(0.975, df=max(N - 1, 1)))

    return g_hat, (float(g_hat - crit * se), float(g_hat + crit * se))


def ppi_plus_estimator(df_all):
    """
    Simple PPI++-style power-tuned estimator using all rows as labeled.
    """
    p_all = df_all["A_prob"].to_numpy(dtype=float)
    H = df_all["H"].to_numpy(dtype=float)
    N = len(df_all)

    if N < 2:
        g, ci = ppi_analytic_estimator(df_all)
        return g, ci, 1.0

    var_p = float(np.var(p_all, ddof=1))
    cov_hp = float(np.cov(H, p_all, ddof=1)[0, 1])

    # Same all-labeled simplification:
    denom = var_p / N + var_p / N
    lam = 0.0 if denom <= 0 or not np.isfinite(denom) else (cov_hp / N) / denom
    lam = float(np.clip(lam, 0.0, 1.0))

    resid = H - lam * p_all
    g_hat = float(lam * p_all.mean() + resid.mean())

    var_resid = float(np.var(resid, ddof=1))
    se = np.sqrt((lam**2) * var_p / N + var_resid / N)
    crit = Z975 if N >= 30 else float(student_t.ppf(0.975, df=max(N - 1, 1)))

    return g_hat, (float(g_hat - crit * se), float(g_hat + crit * se)), lam


def cre_beta_estimator(df_all, alpha=1.0, beta=1.0, S=10000, seed=2025):
    """
    Conjugate CRE using all rows as labeled for descriptive real-data estimate.
    """
    local_rng = np.random.default_rng(seed)

    A = df_all["A_class"].to_numpy(dtype=int)
    H = df_all["H"].to_numpy(dtype=int)
    N = len(A)

    nA1 = int(A.sum())

    mask1 = A == 1
    mask0 = A == 0

    n1 = int(mask1.sum())
    n0 = int(mask0.sum())

    H1 = int(H[mask1].sum())
    H0 = int(H[mask0].sum())

    thetaA = local_rng.beta(alpha + nA1, beta + N - nA1, size=S)
    thetaH1 = local_rng.beta(alpha + H1, beta + n1 - H1, size=S)
    thetaH0 = local_rng.beta(alpha + H0, beta + n0 - H0, size=S)

    g = thetaA * thetaH1 + (1.0 - thetaA) * thetaH0

    return float(g.mean()), (
        float(np.quantile(g, 0.025)),
        float(np.quantile(g, 0.975)),
    )


prev_rows = []

for scope, df_scope in scopes:
    if len(df_scope) == 0 or df_scope["H"].nunique() < 2:
        continue

    g_p, (p_lo, p_hi) = ppi_analytic_estimator(df_scope)
    g_pp, (pp_lo, pp_hi), lam = ppi_plus_estimator(df_scope)
    g_cu, (cu_lo, cu_hi) = cre_beta_estimator(df_scope, alpha=1.0, beta=1.0, seed=SEED)
    g_cj, (cj_lo, cj_hi) = cre_beta_estimator(df_scope, alpha=0.5, beta=0.5, seed=SEED + 1)

    prev_rows.append({
        "scope": scope,
        "N": len(df_scope),
        "subjects": df_scope["subject_id"].nunique(),
        "truth_prevalence": float(df_scope["H"].mean()),
        "AUC": float(roc_auc_score(df_scope["H"], df_scope["A_prob"])),

        "ppi_hat": g_p,
        "ppi_lo": p_lo,
        "ppi_hi": p_hi,

        "ppipp_hat": g_pp,
        "ppipp_lo": pp_lo,
        "ppipp_hi": pp_hi,
        "ppipp_lambda": lam,

        "cre_uniform_hat": g_cu,
        "cre_uniform_lo": cu_lo,
        "cre_uniform_hi": cu_hi,

        "cre_jeffreys_hat": g_cj,
        "cre_jeffreys_lo": cj_lo,
        "cre_jeffreys_hi": cj_hi,
    })

prevalence_df = pd.DataFrame(prev_rows)
prevalence_df.to_csv(OUT_DIR / "prevalence_estimators_realdata_subject_oof_scan_age.csv", index=False)

display(prevalence_df.round(4))

In [ ]:
# ============================================================
# 9) Subject-grouped OOF threshold selection audit
# ============================================================

def oof_threshold_metrics_subject_grouped(df_in, K=5, seed=2025):
    """
    Choose Youden threshold on training subjects and evaluate on held-out subjects.
    This is separate from autorater OOF training; it audits threshold selection leakage.
    """
    df_in = df_in.copy().reset_index(drop=True)

    sgkf = StratifiedGroupKFold(n_splits=K, shuffle=True, random_state=seed)

    rows = []

    for fold, (tr, te) in enumerate(
        sgkf.split(df_in, y=df_in["H"].astype(int), groups=df_in["subject_id"].astype(str)),
        start=1,
    ):
        train_subjects = set(df_in.iloc[tr]["subject_id"].astype(str))
        test_subjects = set(df_in.iloc[te]["subject_id"].astype(str))
        overlap = train_subjects & test_subjects

        if overlap:
            raise RuntimeError(f"Subject overlap in threshold fold {fold}: {len(overlap)}")

        train = df_in.iloc[tr]
        test = df_in.iloc[te]

        thr, _, _ = youden_threshold(train["H"].values, train["A_prob"].values)

        m = binary_metrics(test["H"].values, test["A_prob"].values, threshold=thr)

        rows.append({
            "fold": fold,
            "threshold_train": thr,
            "n_train_rows": len(train),
            "n_test_rows": len(test),
            "n_train_subjects": train["subject_id"].nunique(),
            "n_test_subjects": test["subject_id"].nunique(),
            "subject_overlap": len(overlap),
            **m,
        })

    oof = pd.DataFrame(rows)

    # Leaky full-data threshold for comparison
    thr_full, _, _ = youden_threshold(df_in["H"].values, df_in["A_prob"].values)
    m_full = binary_metrics(df_in["H"].values, df_in["A_prob"].values, threshold=thr_full)

    leaky = pd.DataFrame([{
        "fold": "leaky_full",
        "threshold_train": thr_full,
        "n_rows": len(df_in),
        "n_subjects": df_in["subject_id"].nunique(),
        **m_full,
    }])

    return oof, leaky


threshold_oof, threshold_leaky = oof_threshold_metrics_subject_grouped(use, K=5, seed=SEED)

threshold_oof.to_csv(OUT_DIR / "threshold_oof_subject_grouped.csv", index=False)
threshold_leaky.to_csv(OUT_DIR / "threshold_leaky_full.csv", index=False)

print("Subject-grouped OOF threshold metrics")
display(threshold_oof.round(4))

print("Leaky full-data threshold metrics")
display(threshold_leaky.round(4))

In [ ]:
# ============================================================
# 10) Exchangeability diagnostic: labeled vs unlabeled separability
# ============================================================

def exchangeability_diagnostic(df_in, R=50, label_frac=0.30, seed=2025):
    """
    Randomly mark a labeled subset and test whether labeled vs unlabeled
    is predictable from deployment covariates.

    Since the labels here are sampled randomly by design, AUC near 0.5 is expected.
    This provides a diagnostic template and sanity check.
    """
    local_rng = np.random.default_rng(seed)
    rows = []

    features = ["Age", "A_prob"]

    last_plot_df = None

    for r in range(R):
        df = df_in.copy().reset_index(drop=True)
        n = len(df)
        n_labeled = int(round(label_frac * n))

        labeled_idx = local_rng.choice(n, size=n_labeled, replace=False)
        L = np.zeros(n, dtype=int)
        L[labeled_idx] = 1

        X = df[features].to_numpy(dtype=float)
        yL = L

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            yL,
            test_size=0.35,
            random_state=seed + r,
            stratify=yL,
        )

        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=1000),
        )

        clf.fit(X_train, y_train)
        pL = clf.predict_proba(X_test)[:, 1]
        auc_L = roc_auc_score(y_test, pL)

        rows.append({
            "rep": r,
            "n_labeled": int(n_labeled),
            "N": int(n),
            "label_frac": float(label_frac),
            "propensity_auc": float(auc_L),
        })

        if r == R - 1:
            last_plot_df = pd.DataFrame({
                "L": y_test,
                "pL": pL,
            })

    diag = pd.DataFrame(rows)

    if last_plot_df is not None:
        plt.figure(figsize=(6, 4))
        plt.hist(last_plot_df.loc[last_plot_df["L"] == 1, "pL"], bins=20, alpha=0.6, density=True, label="Labeled")
        plt.hist(last_plot_df.loc[last_plot_df["L"] == 0, "pL"], bins=20, alpha=0.6, density=True, label="Unlabeled")
        plt.xlabel("Predicted P(labeled | covariates)")
        plt.ylabel("Density")
        plt.title("Exchangeability diagnostic: propensity overlap")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.savefig(FIG_DIR / "fig_exchangeability_propensity_overlap.png", dpi=300, bbox_inches="tight")
        plt.show()

    return diag


exchangeability_df = exchangeability_diagnostic(use, R=50, label_frac=0.30, seed=SEED)
exchangeability_df.to_csv(OUT_DIR / "exchangeability_auc_random_labeling.csv", index=False)

display(exchangeability_df.describe().round(4))
display(exchangeability_df.head())

In [ ]:
# ============================================================
# 11) Bootstrap dispersion of Youden threshold
# ============================================================

def threshold_bootstrap(df_in, B=2000, seed=2025):
    local_rng = np.random.default_rng(seed)

    y = df_in["H"].to_numpy(dtype=int)
    p = df_in["A_prob"].to_numpy(dtype=float)
    n = len(df_in)

    rows = []

    for b in tqdm(range(B), desc="Threshold bootstrap"):
        idx = local_rng.integers(0, n, size=n)
        yb = y[idx]
        pb = p[idx]

        if len(np.unique(yb)) < 2:
            continue

        thr, _, _ = youden_threshold(yb, pb)

        # Evaluate threshold on original sample for policy dispersion summary
        m = binary_metrics(y, p, threshold=thr)

        rows.append({
            "rep": b,
            "threshold": thr,
            **m,
        })

    out = pd.DataFrame(rows)
    return out


thr_boot = threshold_bootstrap(use, B=THRESH_BOOT_B, seed=SEED)
thr_boot.to_csv(OUT_DIR / "threshold_bootstrap_samples.csv", index=False)

thr_summary = pd.DataFrame([{
    "B_requested": THRESH_BOOT_B,
    "B_used": len(thr_boot),
    "threshold_mean": float(thr_boot["threshold"].mean()),
    "threshold_lo": float(thr_boot["threshold"].quantile(0.025)),
    "threshold_hi": float(thr_boot["threshold"].quantile(0.975)),
    "ACC_mean": float(thr_boot["ACC"].mean()),
    "ACC_lo": float(thr_boot["ACC"].quantile(0.025)),
    "ACC_hi": float(thr_boot["ACC"].quantile(0.975)),
    "TPR_mean": float(thr_boot["TPR"].mean()),
    "TPR_lo": float(thr_boot["TPR"].quantile(0.025)),
    "TPR_hi": float(thr_boot["TPR"].quantile(0.975)),
    "TNR_mean": float(thr_boot["TNR"].mean()),
    "TNR_lo": float(thr_boot["TNR"].quantile(0.025)),
    "TNR_hi": float(thr_boot["TNR"].quantile(0.975)),
}])

thr_summary.to_csv(OUT_DIR / "threshold_bootstrap_summary.csv", index=False)

plt.figure(figsize=(6, 4))
plt.hist(thr_boot["threshold"], bins=30, density=True, alpha=0.8)
plt.xlabel("Youden threshold")
plt.ylabel("Density")
plt.title("Bootstrap dispersion of Youden threshold")
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_threshold_bootstrap_hist.png", dpi=300, bbox_inches="tight")
plt.show()

display(thr_summary.round(4))

In [ ]:
# ============================================================
# 12) Final output list
# ============================================================

print("Saved outputs in:", OUT_DIR)
for p in sorted(OUT_DIR.glob("*")):
    if p.is_file():
        print("-", p.name)

print("\nSaved figures in:", FIG_DIR)
for p in sorted(FIG_DIR.glob("*")):
    print("-", p.name)

print("\nSaved calibration curves in:", CAL_DIR)
for p in sorted(CAL_DIR.glob("*")):
    print("-", p.name)

## Manuscript wording notes

Use wording like:

> All autorater probabilities used in the PPI and CRE experiments were generated out-of-fold using subject-level grouped splits. All scans from the same ADNI participant were assigned to the same fold, and no subject appeared in both the training and held-out fold for any split.

For the age subset:

> Age-specific analyses used scan-level ages matched by ADNI subject identifier and acquisition date. The 65--70 subset therefore contains scans whose acquisition-level age lies between 65 and 70.

For PPI vs CRE:

> PPI and PPI++ use continuous autorater probabilities, whereas CRE and the binary difference estimator use the thresholded decision \(A=\mathbf{1}\{p\ge0.5\}\). CRE is therefore not intended to universally dominate continuous-score PPI in interval width; it targets the thresholded deployment estimand and gives a posterior over the interpretable \(2\times2\) decision table.